# Assignment 02: Fetching Occurrence Records With pygbif

## BIO597 Spatial Analysis of Biodiversity

This assignment gives you more practice with the workflow from Lab 02.

You will use `pygbif` to search GBIF for several snake species, download a small number of occurrence records, convert those records to `GeoDataFrame` objects, and make simple maps and summaries.

Some cells are partly filled in. You should fill in the missing pieces and run the notebook from top to bottom.

Remember, learning to code is often about learning how to strategically copy, paste, and modify. Look back at Lab 02 when you need a model.


## Species for this assignment

Use these species:

* *Storeria dekayi*
* *Agkistrodon contortrix*
* *Pantherophis guttatus*
* one additional snake species of your choice

For each species, we will ask GBIF for georeferenced records so the results can be mapped.


## 1. Import packages

Import the same packages used in Lab 02.

`species` is used for taxonomic name searches. `occ` is used for occurrence record searches.


In [2]:
# To install pygbif, run pip install pygbif in terminal

from pygbif import species
from pygbif import occurrences as occ

import pandas as pd
import geopandas as gpd

## 2. Search for possible name matches

Use `species.name_suggest()` to search for *Storeria dekayi*.

Fill in the species name. Then inspect the first result.


In [3]:
dekayi_suggestions = species.name_suggest(q="Storeria dekay")
# dekayi_suggestions will have a list of dictionaries
# select the first element of this list here and save it as a new variable called `dekayi_match`

dekayi_match = dekayi_suggestions[0]

## 3. Save the taxon key

Pull the `speciesKey` out of the match result and save it as `dekayi_key`.


In [4]:
dekayi_key = dekayi_match["speciesKey"]
dekayi_key

# Extracts unique species ID

9056579

## 4. Count records before downloading

Use `occ.count()` to count georeferenced GBIF records for *Storeria dekayi*.

This count tells you how many records GBIF has that match your search, not how many you will download in this assignment.


In [5]:
dekayi_count = occ.count(
    taxonKey=dekayi_key,
    isGeoreferenced=True,
)

dekayi_count

49474

## 5. Determine how many _total_ records there are for S. dekayi

## 5. Determine how many _total_ records there are for S. dekayi

The `isGeoreferenced` parameter determines whether occurrences with latlongs are returned.
Make a copy of the call to `occ.count()` as above, but change the `True` to `False`, which
will return only occurrences **without** latlongs. Capture the results in a new variable 
called `dekayi_nolatlong_count` and then add this to `dekayi_count` from the previous cell
to get the total number of records.

In [6]:
# How many total S. Dekayi records are there?

# Count occurrences without lat/long
dekayi_nolatlong_count = occ.count(
    taxonKey=dekayi_key,
    isGeoreferenced=False,
)

total_dekayi_count = dekayi_count + dekayi_nolatlong_count

total_dekayi_count

# Is it possible to view multiple outputs from a single cell (I would like to run the cell and be able to see both dekayi_nolatlong_count and total_dekayi_count)

56547

## 6. Fetch up to 100 records

Use `occ.search()` to fetch occurrence records for *Storeria dekayi*.

Keep `limit=100`. Do not request more than 100 records for a species in this assignment.


In [7]:
dekayi_records = occ.search(
    speciesKey=dekayi_key,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

dekayi_records.keys()

dict_keys(['offset', 'limit', 'endOfRecords', 'count', 'results', 'facets'])

## 7. Turn the records into a table

The occurrence records are stored in the "results" key of the `dekayi_records` dictionary. Convert that list of records to a pandas DataFrame.


In [8]:
dekayi_df = pd.DataFrame(dekayi_records["results"])
dekayi_df.head()

,key,datasetKey,publishingOrgKey,datasetCategory,installationKey,hostingOrganizationKey,publishingCountry,protocol,lastCrawled,lastParsed,...,eventTime,identificationID,occurrenceRemarks,informationWithheld,projectId,dynamicProperties,vitality,lifeStage,identificationRemarks,sex
0,5938064620,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:22:11.912+00:00,...,16:48:33-06:00,745332273,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5938201753,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:48:03.041+00:00,...,14:41:00-06:00,746093157,Body about 0.5 cm wide.,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5938457152,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:36:08.412+00:00,...,04:52:51-06:00,745694353,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5938490464,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:51:42.840+00:00,...,14:00:00-05:00,746922821,First snake of the year !!!,Coordinate uncertainty increased to 28734m at ...,NaN,NaN,NaN,NaN,NaN,NaN
4,5938606080,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:22:15.726+00:00,...,11:27:14-06:00,745678895,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Keep a small set of useful columns

Keep the columns needed for a map and a few simple summaries.

Fill in the latitude column name.


In [9]:
dekayi_small = dekayi_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

dekayi_small.head()

# How can we view all column names?

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode
0,5938064620,"Storeria dekayi (Holbrook, 1839)",-97.122430,33.241408,2026,HUMAN_OBSERVATION,US
1,5938201753,"Storeria dekayi (Holbrook, 1839)",-92.908814,34.619900,2026,HUMAN_OBSERVATION,US
2,5938457152,"Storeria dekayi (Holbrook, 1839)",-86.721496,33.458035,2026,HUMAN_OBSERVATION,US
3,5938490464,"Storeria dekayi (Holbrook, 1839)",-80.744301,35.172939,2026,HUMAN_OBSERVATION,US
4,5938606080,"Storeria dekayi (Holbrook, 1839)",-97.508688,35.238997,2026,HUMAN_OBSERVATION,US


## 9. Convert the table to a GeoDataFrame

Use the longitude and latitude columns to create point geometry. This is the same idea as Lab 01 and Lab 02.


In [10]:
dekayi_gdf = gpd.GeoDataFrame(
    dekayi_small,
    geometry=gpd.points_from_xy(dekayi_small["decimalLongitude"], dekayi_small["decimalLatitude"]),
    crs="EPSG:4326",
)

dekayi_gdf["Species"] = "Storeria dekayi"
dekayi_gdf.head()

,key,scientificName,decimalLongitude,decimalLatitude,year,basisOfRecord,countryCode,geometry,Species
0,5938064620,"Storeria dekayi (Holbrook, 1839)",-97.122430,33.241408,2026,HUMAN_OBSERVATION,US,POINT (-97.12243 33.24141),Storeria dekayi
1,5938201753,"Storeria dekayi (Holbrook, 1839)",-92.908814,34.619900,2026,HUMAN_OBSERVATION,US,POINT (-92.90881 34.6199),Storeria dekayi
2,5938457152,"Storeria dekayi (Holbrook, 1839)",-86.721496,33.458035,2026,HUMAN_OBSERVATION,US,POINT (-86.7215 33.45804),Storeria dekayi
3,5938490464,"Storeria dekayi (Holbrook, 1839)",-80.744301,35.172939,2026,HUMAN_OBSERVATION,US,POINT (-80.7443 35.17294),Storeria dekayi
4,5938606080,"Storeria dekayi (Holbrook, 1839)",-97.508688,35.238997,2026,HUMAN_OBSERVATION,US,POINT (-97.50869 35.239),Storeria dekayi


## 10. Map *Storeria dekayi*

Make an interactive map of the *Storeria dekayi* records.


In [11]:
dekayi_gdf.explore()

## 11. Repeat the workflow for *Agkistrodon contortrix*

Now repeat the same steps for eastern copperhead, *Agkistrodon contortrix*.

This cell should get the name suggestions and save the speciesKey.


In [12]:
contortrix_suggestions = species.name_suggest(q="Agkistrodon contortrix")

# Select the first element from the `contortrix_suggestions` list
contortrix_match = contortrix_suggestions[0]

# Get the `speciesKey`
contortrix_key = contortrix_match["speciesKey"]

contortrix_key

9215881

## 13. Fetch and map *Agkistrodon contortrix*

Write code to fetch up to 100 records with coordinates, convert them to a table, convert that table to a GeoDataFrame, add a `Species` column, and map the result.

Use the same variable names shown in the comments. In `occ.search()`, use `hasCoordinate=True` and `hasGeospatialIssue=False`.


In [13]:
# Create contortrix_records with occ.search().
contortrix_records = occ.search(
    taxonKey=contortrix_key,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

# Create contortrix_df from contortrix_records["results"].
contortrix_df = pd.DataFrame(contortrix_records["results"])

# Create contortrix_small with the columns you want to keep.
contortrix_small = contortrix_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]

# Create contortrix_gdf with gpd.GeoDataFrame().
contortrix_gdf = gpd.GeoDataFrame(
    contortrix_df,
    geometry=gpd.points_from_xy(contortrix_df["decimalLongitude"], contortrix_df["decimalLatitude"]),
    crs="EPSG:4326",
)


# Add a Species column with the name Agkistrodon contortrix.
contortrix_gdf["Species"] = "Agkistrodon contortrix"

# Map contortrix_gdf with .explore().
contortrix_gdf.explore()


## 14. Repeat the workflow for *Pantherophis guttatus*

This time you will do a little more on your own.

First, use `species.name_suggest()` to get the `speciesKey` for *Pantherophis guttatus*.


In [22]:
# Write your code here.
guttatus_suggestions = species.name_suggest(q="Pantherophis guttatus")

guttatus_match = guttatus_suggestions[0]

guttatus_key = guttatus_match["speciesKey"]

guttatus_key
# Is there a way to see if it is running a cell?

2455615

## 16. Fetch and map *Pantherophis guttatus*

Fetch up to 100 georeferenced records and convert them to a GeoDataFrame.

Keep the cell simple. It is fine to copy and modify code from earlier cells.


In [15]:
# Write your code here.
guttatus_records = occ.search(
    taxonKey=guttatus_key,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

guttatus_df = pd.DataFrame(guttatus_records["results"])


guttatus_small = guttatus_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]


guttatus_gdf = gpd.GeoDataFrame(
    guttatus_df,
    geometry=gpd.points_from_xy(guttatus_df["decimalLongitude"], guttatus_df["decimalLatitude"]),
    crs="EPSG:4326",
)



guttatus_gdf["Species"] = "Pantherophis guttatus"


guttatus_gdf.explore()

## 17. Combine the three GeoDataFrames

Use `pd.concat()` to combine your three species GeoDataFrames.

Then inspect the first few rows.


In [16]:
gbif_snakes = pd.concat([guttatus_gdf, contortrix_gdf, dekayi_gdf])

gbif_snakes.head()

,key,datasetKey,publishingOrgKey,datasetCategory,installationKey,hostingOrganizationKey,publishingCountry,protocol,lastCrawled,lastParsed,...,catalogNumber,institutionCode,vitality,eventTime,identificationID,occurrenceRemarks,sex,geometry,Species,identificationRemarks
0,5938135213,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:58:06.727+00:00,...,333348517,iNaturalist,alive,10:30:48-05:00,746112916,NaN,NaN,POINT (-81.39497 30.13512),Pantherophis guttatus,NaN
1,5938250009,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T06:07:09.381+00:00,...,333098901,iNaturalist,dead,22:16:01-05:00,745410959,"Dor, ORCA golf path",NaN,POINT (-80.2785 25.32908),Pantherophis guttatus,NaN
2,5938256663,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:38:29.771+00:00,...,333329391,iNaturalist,NaN,18:32:25-05:00,746056624,NaN,NaN,POINT (-80.45259 27.78972),Pantherophis guttatus,NaN
3,5938387083,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T05:36:00.531+00:00,...,333236865,iNaturalist,NaN,13:58:14-06:00,745797397,NaN,NaN,POINT (-88.01538 30.23198),Pantherophis guttatus,NaN
4,5938619022,50c9509d-22c7-4a22-a47d-8c48425ef4a7,28eb1a3f-1c15-4a95-931a-4af90ecb574d,[CitizenScience],997448a8-f762-11e1-a439-00145eb45e9a,28eb1a3f-1c15-4a95-931a-4af90ecb574d,US,DWC_ARCHIVE,2026-09-10T19:15:51.157+00:00,2026-09-11T06:06:47.494+00:00,...,333571792,iNaturalist,NaN,07:25:23-05:00,746787555,NaN,NaN,POINT (-81.79749 26.2456),Pantherophis guttatus,NaN


## 18. Map all three species together

Use `.explore()` and color by `Species` so you can compare the three species on one map.


In [17]:
gbif_snakes.explore(column="Species", cmap="rainbow")

## 19. Count records by species

Use `groupby()` to count how many records you downloaded for each species.


In [18]:
gbif_snakes.groupby("Species").size()

Species
Agkistrodon contortrix    100
Pantherophis guttatus     100
Storeria dekayi           100
dtype: int64

## 20. Count records by basis of record

The `countryCode` field describes the general type of occurrence record.

Use `groupby()` to count records by `countryCode`.


In [19]:
gbif_snakes.groupby("countryCode").size()

countryCode
MX      2
US    298
dtype: int64

## 21. Choose one additional species

Choose one additional snake species and repeat the workflow.

Your species does not have to be in the local `EasternSnakes` CSV files. It only needs to be a snake species that GBIF can find.

Your code should:

* use `species.name_suggest()`
* save the taxon key
* count georeferenced records
* fetch no more than 100 records with `occ.search()`
* convert the records to a `GeoDataFrame`
* map the records


In [23]:
# Write your code here.

# Repeating for Lampropeltis triangulum
triangulum_suggestions = species.name_suggest(q="Lampropeltis triangulum")

triangulum_match = triangulum_suggestions[0]

triangulum_key = triangulum_match["speciesKey"]

triangulum_count = occ.count(
    taxonKey=triangulum_key,
    isGeoreferenced=True,
)

triangulum_count



22146

In [24]:
triangulum_records = occ.search(
    taxonKey=triangulum_key,
    hasCoordinate=True,
    hasGeospatialIssue=False,
    limit=100,
)

triangulum_df = pd.DataFrame(triangulum_records["results"])


triangulum_small = triangulum_df[
    ["key", "scientificName", "decimalLongitude", "decimalLatitude", "year", "basisOfRecord", "countryCode"]
]


triangulum_gdf = gpd.GeoDataFrame(
    triangulum_df,
    geometry=gpd.points_from_xy(triangulum_df["decimalLongitude"], triangulum_df["decimalLatitude"]),
    crs="EPSG:4326",
)



triangulum_gdf["Species"] = "Lampropeltis triangulum"


triangulum_gdf.explore()

## 22. Written reflection

Answer these questions after running your code.

**Question 1:** Which of your species had the most GBIF records available?

**Your answer:** S. dekayi

**Question 2:** Did the GBIF points look similar to the local CSV points from Assignment 01? Why might GBIF records look different?

**Your answer:** The local CSV points were more evenly distributed, the GBIF records were clustered. I don't where the CSV points came from but I imagine that GBIF concatenates data from multiple sources and methods. Maybe the CSV points were from some sort of systematic survey of presence?

**Question 3:** What is one reason it is useful to count records before downloading or mapping them?

**Your answer:** It is helpful to know how many records are available, the best method for downloading, and what to expect when visualizing a map to catch any discrepancies. 


In [28]:
# Compare records across species with for loop
# Created by consulting Stack Overflow and Gemini

# List of species keys
snakes = [dekayi_key, contortrix_key, guttatus_key, triangulum_key]

# List of species names
snake_names = ["S. dekayi", "A. contortrix", "P. guttatus", "L. triangulum"]

# Dictionary of keys, names
snake_dict = dict(zip(snakes, snake_names))

# Empty list to hold rows
rows = []

for snake_key, species_name in snake_dict.items():
    count = occ.count(
        taxonKey = snake_key,
        isGeoreferenced = True,
    )
    nolatlong_count = occ.count(
        taxonKey = snake_key,
        isGeoreferenced = False,
    )
    total = count + nolatlong_count
    rows.append({
        "Species": species_name,
        "Species key": snake_key,
        "Georeferenced count": count,
        "No lat/long count": nolatlong_count,
        "Total count": total
    })


snake_totals = pd.DataFrame(rows)

print(snake_totals)


         Species  Species key  Georeferenced count  No lat/long count  \
0      S. dekayi      9056579                49474               7073   
1  A. contortrix      9215881                23303               5812   
2    P. guttatus      2455615                10142               3072   
3  L. triangulum      5224503                22146               4347   

   Total count  
0        56547  
1        29115  
2        13214  
3        26493  


## 23. Submit your work

Before submitting, make sure you have run the notebook from top to bottom and answered the written questions.

Commit and push your completed notebook to your class GitHub repository.

Open a terminal window and run these commands to add, commit, and push your notebook:

```
# Go to the labs directory in the course repo
cd ~/BIO597-SpatialBiodiversity/docs/assignments

# Add your changed lab
git add Assignment-02-pygbif.ipynb
git commit -m 'Finished Assignment 02'
git push
```